# 04_1 — Optimisation du modèle et du seuil métier

**Projet 6 OpenClassrooms — Modèle de scoring crédit « Prêt à dépenser »**

**Étape 4 des consignes : « Optimisez le modèle et le seuil métier »**. Résultat attendu : un modèle avec des hyperparamètres optimisés et un seuil de décision ajusté au coût métier.

**Point de départ.** Le notebook 03_1 a comparé 8 modèles avec leurs réglages par défaut. Le meilleur est **LightGBM avec `class_weight="balanced"`** : coût métier de 0,497 en validation croisée (contre 0,807 pour la référence « tout accorder »), seuil optimal de 0,48, AUC de validation de 0,782.

**Objectif.** Chercher les hyperparamètres de LightGBM qui **minimisent directement le coût métier**, puis fixer le seuil de décision final, vérifier la robustesse du modèle, et le confronter **une seule fois** au jeu de test mis de côté depuis l'étape 1.

**Démarche**
1. Recharger les données et se connecter à MLflow.
2. Définir l'espace de recherche des hyperparamètres.
3. Écrire la fonction objectif d'Optuna : validation croisée, coût métier au seuil optimal, un run MLflow par essai.
4. Lancer l'étude et l'analyser.
5. Fixer le seuil final et contrôler la robustesse.
6. Évaluer le modèle final sur le jeu de test.

Toutes les décisions (hyperparamètres, seuil) sont prises sur le **train** en validation croisée. Le **test** ne sert qu'une fois, à la fin, pour mesurer la performance sur des clients jamais vus

Import des bibliotheques, pour Optuna; le modèle sera LightGBM jugé comme plus performant

In [1]:
# -------------------------- Imports --------------------------
import os
import sys

sys.path.append("..")
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"  # masque un message d'information de MLflow 3

import json

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold

sns.set()
optuna.logging.set_verbosity(optuna.logging.WARNING)  # n'affiche que les avertissements, pas une ligne par essai

from src.settings import (
    COST_FN,
    COST_FP,
    FEATURES_USED_FILE,
    ID_COLUMN,
    MLFLOW_EXPERIMENT_NAME,
    MLFLOW_TRACKING_URI,
    TARGET,
    TEST_FILE,
    TRAIN_FILE,
    random_state,
)
from src.modeling_functions import (
    cost_by_threshold,
    cross_validate_oof,
    find_best_threshold,
    scores_at_threshold,
)


## 1. Chargement des données et configuration MLflow


Chargement des modèles. Charger le train uniquement, et la liste des 659 featyres retenues dans l'étape 1. Test n'est pas chargé ici

In [2]:
with open(FEATURES_USED_FILE, "r") as f:
    features_used = json.load(f)

train = pd.read_parquet(TRAIN_FILE)
X = train[features_used].astype("float32")
y = train[TARGET]

print(f"X : {X.shape[0]:,} lignes x {X.shape[1]} features | {X.memory_usage().sum() / 1024**2:.0f} Mo")
print(f"Taux de défaut : {y.mean():.2%} | ratio bons / mauvais clients : {(y == 0).sum() / (y == 1).sum():.1f}")


X : 246,005 lignes x 659 features | 618 Mo
Taux de défaut : 8.07% | ratio bons / mauvais clients : 11.4


## 2. Pourquoi optimiser, et pourquoi Optuna ?

Se connecter à MLFlow, lire les valeurs directement enregistée dans MLFlow avec search runs par l'API, c'est la solution d'automatisation car un futur script d'entrapinement pourra comparer automatiquement le nouevau modèle à l'ancien.

In [3]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
experiment = mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
dataset_abstraction = mlflow.data.from_pandas(
    train.drop(columns=ID_COLUMN), source=str(TRAIN_FILE.name), targets=TARGET, name="train_v1"
)
del train  # X et y suffisent désormais : on libère la mémoire

# Référence à battre : le meilleur run du 03_1, lu directement dans MLflow
reference_run = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.notebook = '03_1_modeles_comparaison' and tags.mlflow.runName = 'lightgbm_balanced'",
    order_by=["start_time DESC"],
    max_results=1,
).iloc[0]
reference_cost = reference_run["metrics.average_test_business_cost"]
reference_threshold = reference_run["metrics.best_threshold"]

print("Expérience :", experiment.name, "| id :", experiment.experiment_id)
print(f"Référence à battre (lightgbm_balanced, 03_1) : coût {reference_cost:.4f} au seuil {reference_threshold:.2f}")


/opt/anaconda3/envs/medef/lib/python3.12/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


Expérience : scoring_credit | id : 1
Référence à battre (lightgbm_balanced, 03_1) : coût 0.4966 au seuil 0.48


## 3. L'espace de recherche des hyperparamètres


Cette section nous permet justifier un choix de méthode. Ici optuna plutôt que GridSearch. Nous faisons un exemple d'évaluation du nécessaire pour recherche les hyperparamètres. Avec environ 9 s par entraînement, la grille donne 78 125 combinaisons, soit à peu près 1 000 heures (environ 40 jours). Avec Optuna, 40 essais prennent à peu près 30 minutes. L'écart parle de lui-même.

On optimise car pour l'instant LightGBM tournait avec ses réglages par défaut, qui sont pensés pour marcher "partout" et non précisement pour notre probleme. Les hyperparamètres recherchent un compromis entre sous et sur apprentissage, notamment pour les paramètres :
- la complexité des arbres (num leaves)
- la vitesse d'apprentissage (learning rate et n estimators)
- l'aléa (subsample, )
- la régularisation (reg lamdba)
Notre objetof n'est pas de maximisuer l'AUC mais le coût métier (ici le minimiser)

Il existe traditionnellement 3 méthodes posssibles :

| Méthode                | Principe                                                 | Limite                                     |
| ---------------------- | -------------------------------------------------------- | ------------------------------------------ |
| **GridSearchCV**       | Teste **toutes** les combinaisons d'une grille           | Explosion combinatoire (voir la cellule)   |
| **RandomizedSearchCV** | Tire **au hasard** N combinaisons                        | Ne tire aucune leçon des essais précédents |
| **Optuna (TPE)**       | **Apprend** des essais passés pour proposer les suivants | Légère complexité de mise en place         |

Comment Optuna apprend ?
LE TPESampler d'Optuna utilise l'algorithme Tree structured Parzen Estimator , une méthode d'optimisation bayésienne adaptée à la recheche d'hyper paramètres. Optuna sépare l'historique des essais en deux groupes selon la qualité de la fonction d'objectif : "bon" et "mauvais". Ensuite il modèle les modélise par densité et ajuste des modèle de mélage de Gausiennes ou des distributions surce deux groupes pour estimer  des nouveaux réglades qui semblent correspondre aux bons. Ici notre critère n'est pas un score standard : pour chaque essai, il faut chercher le seuil optimal sur les prédictions hors, puis calculer le coût.






In [4]:
grid = {
    "n_estimators": [200, 400, 600, 800, 1000],
    "learning_rate": [0.01, 0.02, 0.05, 0.1, 0.2],
    "num_leaves": [15, 31, 63, 127, 255],
    "min_child_samples": [10, 20, 50, 100, 200],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.3, 0.5, 0.7, 0.9, 1.0],
    "reg_lambda": [0, 0.1, 1, 10, 100],
}
n_combinations = np.prod([len(values) for values in grid.values()])
n_folds = 5
fit_time = reference_run["metrics.average_fit_time_s"]  # temps moyen d'un entraînement au 03_1, lu dans MLflow

grid_hours = n_combinations * n_folds * fit_time / 3600
print(f"Grille complète : {n_combinations:,} combinaisons x {n_folds} plis x {fit_time:.0f} s = {grid_hours:,.0f} heures ({grid_hours / 24:,.0f} jours)")

n_trials = 40
print(f"Optuna, {n_trials} essais : {n_trials * n_folds * fit_time / 60:.0f} minutes environ (plus si les essais ont beaucoup d'arbres)")


Grille complète : 78,125 combinaisons x 5 plis x 9 s = 1,013 heures (42 jours)
Optuna, 40 essais : 31 minutes environ (plus si les essais ont beaucoup d'arbres)


La cellue suivante définit l'espace de recherche, elle n'entraîne rien. Elle affiche seulement le modèle contruit avec les réglages par défaut.

Le choix d'un nomnre d'arbbres successifs trop important (2000) est problématique à cause du budget de calcul. L'espace de recheche est un compromis entre liberté laissée à Optuna et temps disponible.

In [5]:
FIXED_PARAMS = {
    "class_weight": "balanced",  # retenu au 03_1
    "subsample_freq": 1,  # sans lui, LightGBM ignore subsample
    "random_state": random_state,
    "n_jobs": -1,
    "verbose": -1,
}


def suggest_params(trial: optuna.Trial) -> dict:
    """Hyperparamètres proposés par Optuna pour un essai."""
    return {
        "n_estimators": trial.suggest_int("n_estimators", 100, 800, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 8, 128, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 300, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.2, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
    }


def build_model(params: dict) -> LGBMClassifier:
    return LGBMClassifier(**FIXED_PARAMS, **params)


# Réglages par défaut de LightGBM (ceux du 03_1), dans l'espace de recherche : servira de premier essai
DEFAULT_PARAMS = {
    "n_estimators": 100,
    "learning_rate": 0.1,
    "num_leaves": 31,
    "min_child_samples": 20,
    "subsample": 1.0,
    "colsample_bytree": 1.0,
    "reg_alpha": 1e-3,
    "reg_lambda": 1e-3,
}
build_model(DEFAULT_PARAMS)


,class_weight,'balanced'
,subsample_freq,1
,reg_alpha,0.001
,reg_lambda,0.001
,random_state,42
,n_jobs,-1
,verbose,-1
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1


## 4. La fonction objectif : le coût métier en validation croisée


C'est le coeur du noteboo,. Optuna ne connaît rien au crédit (à la fonction métier) : il appelle une fonction externe pré-établie, reçoit un nombre et cherche à le rendre le plus petit possible. tout notre savoir faire métier est donc dans cette fonction.

La fonction Objectif


Cette fonction, à chaque essai :
1. Optuna propose des hyperparamètres (suggest marams).
2. Validation croisée en 5 plus avec les mêmes plis qu'au précédent (03-1 (même random-state)) On obtient les prédictions hors pli.
3. Recherche du seuil qui minimise le coût sur ces prédictions.
4. Calcul du coût pli par pli à ce seuil : moye,,e et écart type.
5. Enregistrement dans un run MLflow entrant.
6. Renvoi du coût métier moye, à Optuna : c'est le nombre qu'il cherche à minimuser.

Précisions :
On optimise le couple (hyper-paramètres - seuil); pas les hyperparamètres seuls. Chaque essai est jugé à son meilleur seuil. Sans cela, un modèle aux probabilités "décalées" serait injustement pénalisé au seuil de 0,5 alors qu'il serait excellent à 0,3. Aussi, pourquoi les mêmes plis qu'au 03-01 ? Simplement Pour comparer à areme égales acvec les références (0,497). Une différence de coput viendra des hyperparamètres, pas du hasard du découpage.

Dans la section suivant, nous allons apporté de la nouveauté pour notre usage d'MLFlow ; on ouvrira un run parent ("l'étude Optuna"), o^chaque essai crée un run enfant à l'intérieur. Un essai = 5 entraînements (un par pli). Au notebook précédent nous avions définis que LightGBM prenait environ 9secondes? Avec ys d'abres, le temps serait davantage 15 à 20 secondes. Un essai prend donc environ 1 minuten et 40 essais prendrait 30 à 45 min. Il faut faire un arbitrage entre le budget de caclul (temps et ressource de l'ordinateur) avec le choix de se passer volontairement de l'optimum absolu.




In [ ]:
n_splits = 5
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)  # mêmes plis qu'au 03_1


def objective(trial: optuna.Trial) -> float:
    """Un essai Optuna = un jeu d'hyperparamètres évalué en validation croisée = un run MLflow enfant."""
    params = suggest_params(trial)

    with mlflow.start_run(
        run_name=f"trial_{trial.number:02d}",
        nested=True,
        tags={"etape": "4 - optimisation", "notebook": "04_1_optimisation_seuil", "optuna_trial": trial.number},
    ):
        oof_proba, val_indices, fold_scores = cross_validate_oof(build_model(params), X, y, cv)
        best_threshold, _ = find_best_threshold(y, oof_proba)
        scores = scores_at_threshold(y, oof_proba, val_indices, best_threshold)

        metrics = {
            "average_train_roc_auc": np.mean(fold_scores["train_roc_auc"]),
            "average_test_roc_auc": np.mean(fold_scores["test_roc_auc"]),
            "test_roc_auc_std": np.std(fold_scores["test_roc_auc"]),
            "best_threshold": best_threshold,
            "average_fit_time_s": np.mean(fold_scores["fit_time"]),
            **scores,
        }
        metrics["overfit_gap_roc_auc"] = metrics["average_train_roc_auc"] - metrics["average_test_roc_auc"]

        mlflow.log_params(params)
        mlflow.log_metrics(metrics)

    # Informations gardées aussi côté Optuna, pour l'analyse de l'étude (section 6)
    trial.set_user_attr("best_threshold", best_threshold)
    trial.set_user_attr("average_test_roc_auc", metrics["average_test_roc_auc"])
    trial.set_user_attr("overfit_gap_roc_auc", metrics["overfit_gap_roc_auc"])

    return metrics["average_test_business_cost"


## 5. Lancement de l'étude Optuna

Après une erreur de chargement des données sur 40 trials, choix de cloturer à 25 essais? Chomme chaque essai était enregistré dans MLFlow au fur et à mesure, rien n'a été perdu. 

In [9]:
# -------------------------- Clôture de l'étude interrompue à 25 essais --------------------------
completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
best = study.best_trial  # Optuna ne tient compte que des essais terminés

parent_run = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.mlflow.runName = 'optuna_lightgbm_balanced'",
    order_by=["start_time DESC"],
    max_results=1,
).iloc[0]

with mlflow.start_run(run_id=parent_run["run_id"]):  # on rouvre le run parent pour y écrire le résumé
    mlflow.set_tag("note_interruption", f"étude arrêtée manuellement après {len(completed)} essais (temps de calcul)")
    mlflow.log_param("n_trials_completed", len(completed))
    mlflow.log_params({f"best_{name}": value for name, value in best.params.items()})
    mlflow.log_metrics({
        "best_business_cost": best.value,
        "best_threshold": best.user_attrs["best_threshold"],
        "best_test_roc_auc": best.user_attrs["average_test_roc_auc"],
        "best_overfit_gap_roc_auc": best.user_attrs["overfit_gap_roc_auc"],
        "gain_vs_reference": reference_cost - best.value,
    })

print(f"{len(completed)} essais terminés | meilleur : n°{best.number} | coût {best.value:.4f} "
      f"(référence 03_1 : {reference_cost:.4f}) | seuil {best.user_attrs['best_threshold']:.2f}")
best.params


25 essais terminés | meilleur : n°17 | coût 0.4873 (référence 03_1 : 0.4966) | seuil 0.48


{'n_estimators': 650,
 'learning_rate': 0.02140362201994293,
 'num_leaves': 37,
 'min_child_samples': 180,
 'subsample': 0.9987995375430284,
 'colsample_bytree': 0.5829387158778291,
 'reg_alpha': 1.5545896925729108,
 'reg_lambda': 0.0565831061129996}

## 6. Analyse de l'étude

## 7. Modèle final : courbe coût / seuil et choix du seuil

## 8. Robustesse du modèle

## 9. Évaluation unique sur le jeu de test

## 10. Conclusion